# Universo dei semiconduttori (Bloomberg)

Esplora i dati puliti da `10_clean_bloomberg.py` e i grafici di `11_eda_bloomberg.py`. Prima di aprirlo:

1. copia i file del terminale in `data/bloomberg/raw/` (almeno `prices.parquet`);
2. esegui `uv run python 10_clean_bloomberg.py` e `uv run python 11_eda_bloomberg.py`.

I dati Bloomberg non vanno su GitHub (licenza): restano sul tuo computer.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
from IPython.display import Image, display
from src.data import read_weekly

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
CLEAN, EDA = ROOT / "data/bloomberg/clean", ROOT / "data/bloomberg/eda"
close = read_weekly(CLEAN / "close.csv").droplevel("date_open")
companies = pd.read_csv(CLEAN / "companies.csv", index_col=0)
features = pd.read_parquet(CLEAN / "features.parquet")
checks = pd.read_csv(CLEAN / "validation/checks.csv")
print(f"{len(companies)} società, {close.shape[0]} settimane, {len(features):,} righe di features")

## 1. Le società

Una riga per società: borse unite, primo e ultimo prezzo, se è uscita dal mercato (acquisita, fusa o fallita), rendimento e volatilità annui, massimo drawdown.

In [ ]:
companies.sort_values("market_cap_bn", ascending=False)

## 2. I controlli sui dati

Nessun FAIL: gli avvisi (WARN) sono spiegati in `docs/bloomberg.md`.

In [ ]:
print(checks["result"].value_counts().to_dict())
checks[(checks["result"] != "PASS") & (checks["check"] != "outliers")]

## 3. Una società: prezzo settimanale e features

Cambia `TICKER` (per esempio `"AMD"`, `"INTC"`, `"9966620D"` per Wolfspeed). Le features di una settimana usano solo i prezzi fino a quel venerdì.

In [ ]:
TICKER = "NVDA"
px.line(close[TICKER].dropna(), log_y=True, title=f"{TICKER}: chiusura settimanale aggiustata (scala log)").show()
f = features.xs(TICKER, level="company").droplevel("date_open")
f[[c for c in f.columns if not c.endswith("_z")]].tail()

In [ ]:
px.line(f[["momentum_52_4", "volatility_26w", "from_high_52w"]], title=f"{TICKER}: tre features nel tempo").show()

## 4. Una settimana: tutte le società a confronto (z-score)

Ogni feature è tagliata all'1% e al 99% e standardizzata dentro la settimana: 0 è la società media di quella settimana.

In [ ]:
WEEK = "2020-03-20"
week = features.xs(pd.Timestamp(WEEK), level="date_close").droplevel("date_open")
week[[c for c in week.columns if c.endswith("_z")]].sort_values("momentum_52_4_z", ascending=False).round(2)

## 5. I grafici dell'analisi esplorativa

In [ ]:
for name in ["universe", "coverage", "survivors", "concentration", "benchmarks", "returns", "feature_correlation"]:
    display(Image(filename=EDA / f"{name}.png", width=900))